<a href="https://colab.research.google.com/github/arturexxx/Alura-AgenteIA-RAG-PDF/blob/main/notebooks/Agente_RETCC_Alura_Desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ==========================================================================================================
# PROYECTO: AGENTE DE IA PARA CONSULTAS DEL REGISTRO NACIONAL DE TRABAJADORES DE CONSTRUCCIÓN CIVIL (RETCC)
# ==========================================================================================================
# Descripción:
# Este proyecto implementa un sistema RAG (Retrieval-Augmented
# Generation) capaz de comprender el contenido de múltiples
# documentos PDF relacionados con el Registro Nacional de
# Trabajadores de Construcción Civil (RETCC) en el Estado Peruano.
#
# El agente podrá responder preguntas sobre:
# - Reglamento del RETCC.
# - Requisitos para la inscripción del Carné RETCC.
#
# Documentos utilizados:
# 1. Reglamento_Inscripcion.pdf
# 2. Requisitos_Inscripcion_Carnet_RETCC.pdf
#
# Tecnologías:
# - Python
# - Google Colab
# - LangChain
# - Hugging Face (Embeddings)
# - FAISS (Base Vectorial)
# - Google Gemini (LLM)
#
# Objetivo:
# Construir un asistente inteligente que consulte la información
# de los documentos y responda preguntas utilizando RAG.
# ============================================================

print("🚀 Iniciando el proyecto: Agente IA RAG para documentos del RETCC")

🚀 Iniciando el proyecto: Agente IA RAG para documentos del RETCC


In [ ]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-google-genai \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-dotenv \
    gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [17]:
from google.colab import files

archivos_subidos = files.upload()

Saving Requisitos_Inscripcion_Carnet_RETCC.pdf to Requisitos_Inscripcion_Carnet_RETCC.pdf
Saving Reglamento_Incripcion.pdf to Reglamento_Incripcion.pdf


In [18]:
lista_pdfs = list(archivos_subidos.keys())

print(lista_pdfs)

['Requisitos_Inscripcion_Carnet_RETCC.pdf', 'Reglamento_Incripcion.pdf']


In [26]:
from langchain_community.document_loaders import PyPDFLoader

documentos = []

for pdf in lista_pdfs:
    print(f"Cargando: {pdf}")

    loader = PyPDFLoader(pdf)
    docs = loader.load()

    documentos.extend(docs)

print("\n-----------------------------")
print(f"Total de páginas cargadas: {len(documentos)}")

Cargando: Requisitos_Inscripcion_Carnet_RETCC.pdf
Cargando: Reglamento_Incripcion.pdf

-----------------------------
Total de páginas cargadas: 9


In [28]:
import re

def limpiar_texto(texto: str) -> str:
    """
    Limpia el texto extraído del PDF sin eliminar información importante.
    """

    # Reemplaza espacios especiales
    texto = texto.replace("\xa0", " ")

    # Une palabras separadas por guion al final de una línea
    # Ejemplo: "traba-\njador" -> "trabajador"
    texto = re.sub(r"(\w)-\n(\w)", r"\1\2", texto)

    # Reemplaza saltos simples de línea por espacios
    texto = re.sub(r"(?<!\n)\n(?!\n)", " ", texto)

    # Reduce múltiples saltos de línea
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    # Reduce múltiples espacios
    texto = re.sub(r"[ \t]+", " ", texto)

    # Elimina espacios al inicio y final
    return texto.strip()


print("Función de limpieza creada correctamente.")

Función de limpieza creada correctamente.


In [29]:
documentos_limpios = []

for documento in documentos:
    documento.page_content = limpiar_texto(documento.page_content)
    documentos_limpios.append(documento)

print(f"Páginas limpiadas: {len(documentos_limpios)}")

Páginas limpiadas: 9


In [30]:
print(documentos_limpios[0].page_content[:2000])

Servicios del RETCC en las sedes del MAC de Lima Metropolitana. + Desde el día 13 de octubre de 2020 se etectuado la reapertura de los servicios de las MAC de Lima Sur y Lima Este; asimismo con fecha el 21 de marzo del 2022 se iniciaron as atenciones en el centro MAC Lima Norte, cabe precisar que en aichos MAC se encuentra brindaco el servicio de RETCC, por parte del personal de la ¡Subarrección de Regssiros Generales (en adelante SDRG), pudiendo acercarse los acminisrados de manera presencial sin necesidad de sacar ca para realizar su trámite de inscripción o renovación o en caso contra a recoger su camé RETCC que haya sido aprobado de manera vitual Requisitos para la inscripción del carné de RETCC + Deberá extibirsu DNIfísico o su Hoja CA, asimismo deberá presentar su Cercado de Antecedentes Penales en caso de homonimia. Requisitos para la renovación del carné de RETCC: 1. Solicitud según formato, 2. Copia simple de cercado o constancia de capacitación emitida por SENCICO y otras ent

In [31]:
documentos_limpios = [
    documento
    for documento in documentos_limpios
    if documento.page_content.strip()
]

print(f"Páginas con contenido: {len(documentos_limpios)}")

Páginas con contenido: 9


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " ",
        ""
    ]
)

print("Divisor configurado correctamente.")

Divisor configurado correctamente.


In [34]:
fragmentos = text_splitter.split_documents(documentos_limpios)

print(f"Páginas procesadas: {len(documentos_limpios)}")
print(f"Fragmentos generados: {len(fragmentos)}")

Páginas procesadas: 9
Fragmentos generados: 35


In [35]:
for indice, fragmento in enumerate(fragmentos):
    pagina_langchain = fragmento.metadata.get("page", 0)

    fragmento.metadata["chunk_id"] = indice
    fragmento.metadata["pagina_real"] = pagina_langchain + 1

print("Metadatos de trazabilidad agregados.")

Metadatos de trazabilidad agregados.


In [36]:
cantidad_mostrar = min(5, len(fragmentos))

for indice in range(cantidad_mostrar):
    fragmento = fragmentos[indice]

    print("=" * 80)
    print(f"Fragmento: {fragmento.metadata['chunk_id']}")
    print(f"Página: {fragmento.metadata['pagina_real']}")
    print(f"Caracteres: {len(fragmento.page_content)}")
    print("-" * 80)
    print(fragmento.page_content[:700])
    print()

Fragmento: 0
Página: 1
Caracteres: 914
--------------------------------------------------------------------------------
Servicios del RETCC en las sedes del MAC de Lima Metropolitana. + Desde el día 13 de octubre de 2020 se etectuado la reapertura de los servicios de las MAC de Lima Sur y Lima Este; asimismo con fecha el 21 de marzo del 2022 se iniciaron as atenciones en el centro MAC Lima Norte, cabe precisar que en aichos MAC se encuentra brindaco el servicio de RETCC, por parte del personal de la ¡Subarrección de Regssiros Generales (en adelante SDRG), pudiendo acercarse los acminisrados de manera presencial sin necesidad de sacar ca para realizar su trámite de inscripción o renovación o en caso contra a recoger su camé RETCC que haya sido aprobado de manera vitual Requisitos para la inscripción del carné 

Fragmento: 1
Página: 1
Caracteres: 844
--------------------------------------------------------------------------------
. Requisitos para la renovación del carné de RETCC: 1. Sol